In [46]:
import numpy as np
import matplotlib.pyplot as plt
from dataclasses import dataclass
from enum import IntEnum

In [47]:
# Using ISO 8855 coordinate system where +x is forward,
# +y is left, and +z is up.

# Vehicle State Vector [x, y, v_x, v_y, theta, omega]
#
# x, y: Global position on the track in meters
#
# v_x, v_y: Velocity of car in its own local frame, m/s
# (v_x is forward speed, v_y is sideways slide speed)
#
# theta: Heading angle, radians. The direction the car's nose
# is pointing relative to the world.
#
# omega: Yaw rate, radians/second. How fast the car is
# spinning around its center. 
# omega > 0 means counter clockwise (left) turn.

# State vector index mapping
class StateIdx(IntEnum):
    X = 0
    Y = 1
    VX = 2
    VY = 3
    THETA = 4
    OMEGA = 5

def initialize_state(v_x_initial_kph: float) -> np.ndarray:
    """Initialize the 6-DoF state vector with starting v_x converted to m/s"""
    v_x_initial_mps = v_x_initial_kph / 3.6 # Convert kph to m/s
    
    # State: [x, y, v_x, v_y, theta, omega]
    state = np.array([
        0.0,              # x (starting at origin)
        0.0,              # y
        v_x_initial_mps,  # v_x (forward velocity)
        0.0,              # v_y (no initial sideways sliding)
        0.0,              # theta (pointing straight along x-axis)
        0.0               # omega (no initial spin/yaw rate)
    ], dtype=np.float64)
    
    return state

# Test with starting speed of 150 kph
test_state = initialize_state(150.0)
print("Initial State Vector:")
print(test_state)

Initial State Vector:
[ 0.          0.         41.66666667  0.          0.          0.        ]


In [48]:
@dataclass(frozen=True)
class Vehicle:
    # Mass and Inertia
    mass: float         # curb weight + driver (kg)
    i_z: float          # yaw moment of inertial (kg*m^2)
        
    # Geometry. All values in meters.
    wheelbase: float    # distance between front and rear axles
    track_width: float  # distance between centers of left and right tires on the same axle
    cg_to_front: float  # distance from Center of Gravity (CoG) to front axle
    cg_to_rear: float   # distance from CoG to rear axle
    cg_height: float    # height of CoG above ground
        
    # Tire parameters, N/rad (Pacejka simplified stiffness rates)
    cf: float    # front tire cornering stiffness
    cr: float    # rear tire cornering stiffness
    
    # Aerodynamic lift area/downforce area Cl*A (m^2)
    cl_area: float
        
    @property
    def total_length(self) -> float:
        return self.cg_to_front + self.cg_to_rear

In [49]:
# Estimated constants for a 2017 Toyota Sienna SE
sienna = Vehicle(
    mass=2107.0,      # kg, curb weight of 4480lbs + 165lb driver
    i_z=3720.0,       # estimated yaw inertia for large minivan
    wheelbase=3.03,   # meters, ~119.3 inches
    track_width=1.72, # meters, ~67.7 in
    cg_to_front=1.45, # front-heavy bias, ~1.45m from front axle
    cg_to_rear=1.58,  # remaining distance to rear axle
    cg_height=0.66,   # 26.15 in, from NHTSA Rollover Stability Measurements Report 2022 
    cf=80000.0,
    cr=85000.0,
    cl_area=0.1       # positive: slight aerodynamic lift
)

print(sienna)

Vehicle(mass=2107.0, i_z=3720.0, wheelbase=3.03, track_width=1.72, cg_to_front=1.45, cg_to_rear=1.58, cg_height=0.66, cf=80000.0, cr=85000.0, cl_area=0.1)


In [50]:
f1 = Vehicle(
    mass=798.0,       # kg, minimum regulated dry mass w/ driver
    i_z=1200.0,
    wheelbase=3.6,    # max regulated length
    track_width=2.0,  # max overall width
    cg_to_front=1.8,  # roughly 50/50 weight distribution
    cg_to_rear=1.8,
    cg_height=0.25,   # extremely low CoG
    cf=180000.0,
    cr=210000.0,
    cl_area=-3.5      # negative: massive downforce
)

print(f1)

Vehicle(mass=798.0, i_z=1200.0, wheelbase=3.6, track_width=2.0, cg_to_front=1.8, cg_to_rear=1.8, cg_height=0.25, cf=180000.0, cr=210000.0, cl_area=-3.5)


In [51]:
def calculate_slip_angles(state: np.ndarray, delta: float, vehicle: Vehicle):
    """
    Returns front and rear slip angles in radians.
    delta: Steering angle at front wheels in radians.
    delta > 0: steering to the left (+y).
    slip angle alpha > 0: tire is pointed further left than its velocity vector.
    """
    v_x = state[StateIdx.VX]
    v_y = state[StateIdx.VY]
    omega = state[StateIdx.OMEGA]
    
    # Avoid division by zero if car is stopped
    if v_x < 0.1:
        return 0.0, 0.0
    
    # Using bicycle model with delta pointing in front wheel direction.
    # Front slip angle = steering angle minus velocity angle at front axle
    alpha_f = delta - np.arctan2(v_y + vehicle.cg_to_front * omega, v_x)
    
    # Assume the rear wheel is aligned with car centerline 
    # (rear wheels don't steer), so delta at the rear wheel = 0.
    alpha_r = -np.arctan2(v_y - vehicle.cg_to_rear * omega, v_x)
    
    return alpha_f, alpha_r
    

In [52]:
def calculate_normal_forces(v_x: float, vehicle: Vehicle):
    """
    Returns dynamic vertical load on front and rear axles (Newtons).
    Includes static weight distribution + speed-dependent 
    aerodynamic downforce/lift.
    """
    g = 9.81
    rho = 1.225 # air density at sea level (kg/m^3)
    
    # Static axle weight split
    f_z_front_static = vehicle.mass * g * (vehicle.cg_to_rear / vehicle.wheelbase)
    f_z_rear_static = vehicle.mass * g * (vehicle.cg_to_front / vehicle.wheelbase)
    
    # Dynamic aerodynamic force (F_aero = 0.5 * rho * ClA * v^2)
    # Split 50/50 between front and rear axles
    f_aero_axle = (0.5 * rho * vehicle.cl_area * (v_x**2)) / 2.0
    
    f_z_front = max(0.0, f_z_front_static - f_aero_axle)
    f_z_rear = max(0.0, f_z_rear_static - f_aero_axle)
    
    return f_z_front, f_z_rear
      


In [53]:
def calculate_pacejka_lateral_force(alpha: float, f_z: float, c_alpha: float) -> float:
    """
    Returns lateral tire force F_y (Newtons).
    alpha: slip angle (rad)
    f_z: normal load (N)
    c_alpha: cornering stiffness parameter
    """
    if f_z <= 0.0:
        return 0.0
    
    # Pacejka coefficients
    # B: stiffness factor
    # C: shape factor
    # D: peak factor
    # E: curvature factor
    
    # Standard Magic Formula shape parameters for dry asphalt
    C = 1.30      #for lateral force
    E = -1.0
    mu = 1.1      #peak friction coefficient (mu ~ 1.0 - 1.6)
    D = mu * f_z  #peak available force
    
    # Avoid division by zero
    if D <= 0.0:
        return 0.0
    
    B = c_alpha / (C * D)
    
    # Pacejka Magic Formula
    f_y = D * np.sin(C * np.arctan(B * alpha - E * (B * alpha - np.arctan(B * alpha))))
    
    return f_y
    

In [54]:
# Quick test script

test_state = initialize_state(150.0)  # 150 km/h initial speed
delta_test = np.radians(2.0)  # 2 degree steering wheel input

# Calculate for Sienna
s_alpha_f, s_alpha_r = calculate_slip_angles(test_state, delta_test, sienna)
s_fz_f, s_fz_r = calculate_normal_forces(test_state[StateIdx.VX], sienna)

# Calculate for F1
f1_alpha_f, f1_alpha_r = calculate_slip_angles(test_state, delta_test, f1)
f1_fz_f, f1_fz_r = calculate_normal_forces(test_state[StateIdx.VX], f1)

print("Sienna at 150 km/h")
print(f"front slip: {np.degrees(s_alpha_f):.2f} deg")
print(f"rear slip: {np.degrees(s_alpha_r):.2f} deg")
print(f"front axle normal force: {s_fz_f:.1f} N")
print(f"rear axle normal force: {s_fz_r:.1f} N\n")

print("F1 car at 150 km/h")
print(f"front slip: {np.degrees(f1_alpha_f):.2f} deg")
print(f"rear slip: {np.degrees(f1_alpha_r):.2f} deg")
print(f"front axle normal force: {f1_fz_f:.1f} N")
print(f"rear axle normal force: {f1_fz_r:.1f} N\n")

# Calculate lateral tire forces generated at 2 degrees slip
s_fy_f = calculate_pacejka_lateral_force(s_alpha_f, s_fz_f, sienna.cf)
s_fy_r = calculate_pacejka_lateral_force(s_alpha_r, s_fz_r, sienna.cr)

f1_fy_f = calculate_pacejka_lateral_force(f1_alpha_f, f1_fz_f, f1.cf)
f1_fy_r = calculate_pacejka_lateral_force(f1_alpha_r, f1_fz_r, f1.cr)

print("Lateral tire forces")
print(f"Sienna front F_y: {s_fy_f:.1f} N")
print(f"Sienna rear F_y: {s_fy_r:.1f} N")
print(f"F1 Car front F_y: {f1_fy_f:.1f} N")
print(f"F1 Car rear F_y: {f1_fy_r:.1f} N")

Sienna at 150 km/h
front slip: 2.00 deg
rear slip: -0.00 deg
front axle normal force: 10725.1 N
rear axle normal force: 9838.3 N

F1 car at 150 km/h
front slip: 2.00 deg
rear slip: -0.00 deg
front axle normal force: 5775.1 N
rear axle normal force: 5775.1 N

Lateral tire forces
Sienna front F_y: 2765.6 N
Sienna rear F_y: 0.0 N
F1 Car front F_y: 5098.9 N
F1 Car rear F_y: 0.0 N
